<a href="https://colab.research.google.com/github/Al3xMR/Al3xMR/blob/main/Sistema_de_Recuperaci%C3%B3n_de_Informaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema de Recuperación de Información

## Integrantes del grupo

- Riki Guallichico
- Kevin Martínez

## Objetivo

Diseñar e implementar un sistema de recuperación de información que indexe un conjunto de documentos en texto plano y permita ejecutar consultas de texto libre utilizando el modelo vectorial con vectores binarios y ponderación TF-IDF, y el modelo probabilístico BM25. El sistema debe permitir evaluar la calidad de los resultados utilizando métricas estándar como precision y recall.

# Implementación

## a. Construcción del índice

- Leer un corpus de documentos en texto plano.
- Procesamiento básico: tokenización, normalización y remoción de stopwords.
- Construcción de un índice invertido que almacene, para cada término, los documentos en los que aparece y su frecuencia.

### Leer un corpus de documentos en texto plano

In [ ]:
!pip install pandas nltk scikit-learn

In [ ]:
import pandas as pd

In [ ]:
# Descargar corpus
!curl -L -o "amazon-fine-food-reviews.zip" https://www.kaggle.com/api/v1/datasets/download/snap/amazon-fine-food-reviews

In [ ]:
import zipfile
import pandas as pd

zip_path = './amazon-fine-food-reviews.zip'

with zipfile.ZipFile(zip_path) as z:
    with z.open('Reviews.csv') as f:
        df = pd.read_csv(f)

df

In [ ]:
# Cargar corpus
newsgroupsdocs = pd.DataFrame(df[:5000], columns=["Text"])
#newsgroupsdocs = pd.DataFrame(df, columns=["Text"]) # 568454 filas
newsgroupsdocs

### Procesamiento básico: tokenización, normalización y remoción de stopwords.

Normalización y tokenización


In [ ]:
import re
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

In [ ]:
def clean_text(doc):
    """
    Limpia un texto eliminando etiquetas HTML, puntuación básica y caracteres especiales.
    """
    if not isinstance(doc, str):
        return doc

    doc = re.sub(r'<.*?>', ' ', doc)
    doc = doc.replace('\n', ' ').replace('\t', ' ')
    doc = doc.replace('.', ' ').replace(',', '').replace('(', '').replace(')', '')
    doc = doc.replace('"', '').replace("'", '').replace('\x08', '')
    doc = doc.replace('*', '')
    doc = doc.replace('?', '').replace('¿', '').replace('¡', '').replace('!', '')
    doc = doc.replace(':', '').replace(';', '').replace('/', '')
    doc = doc.replace('[', '').replace(']', '').replace('{', '').replace('}', '')
    doc = doc.replace('@', ' ').replace('#', ' ').replace('$', ' ').replace('%', ' ')
    doc = re.sub(r'\!+', '', doc)
    doc = re.sub('-', ' ', doc)
    doc = re.sub(r'\s+', ' ', doc).strip()
    return doc

In [ ]:
# Limpiar, pasar a minúsculas y tokenizar
newsgroupsdocs["Text"] = newsgroupsdocs["Text"].str.lower()
newsgroupsdocs["Text"] = newsgroupsdocs["Text"].apply(clean_text)
newsgroupsdocs["tokens"] = newsgroupsdocs["Text"].str.split()

In [ ]:
newsgroupsdocs

In [ ]:
# Definir stopwords en inglés
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

In [ ]:
# Eliminar stopwords
newsgroupsdocs["tokens"] = newsgroupsdocs["tokens"].apply(
    lambda doc: [t for t in doc if t not in stop_words]
)

In [ ]:
# Tomar todos los documentos ya tokenizados
tokens_all = newsgroupsdocs["tokens"]
tokens_all

In [ ]:
# Aplicar stemming a cada documento
stemmer = PorterStemmer()
documents = [[stemmer.stem(t) for t in doc] for doc in tokens_all]

In [ ]:
# Reconstruir textos desde stems
docs_as_text = [" ".join(doc) for doc in documents]

### Construcción de un índice invertido que almacene, para cada término, los documentos en los que aparece y su frecuencia.

In [ ]:
from collections import defaultdict, Counter

In [ ]:
# Crear índice invertido
inverted_index = defaultdict(dict)

In [ ]:
# Iterar sobre documentos
for doc_id, doc_tokens in enumerate(documents):
    # Contar frecuencia de términos en el documento
    term_freqs = Counter(doc_tokens)
    for term, freq in term_freqs.items():
        inverted_index[term][doc_id] = freq

In [ ]:
inverted_index_df = pd.DataFrame([
    {"term": term, "doc_id": doc_id, "freq": freq}
    for term, postings in inverted_index.items()
    for doc_id, freq in postings.items()
])

In [ ]:
# Mostrar primeras filas
inverted_index_df.head(10)

## b. Modelo de recuperación

- Implementar recuperación basada en similitud Jaccard utilizando vectores binarios
- Implementar recuperación basada en similitud de coseno utilizando TF-IDF
- Implementar recuperación con BM25.
- Permitir la ejecución de consultas de texto libre.
- Mostrar un ranking de documentos ordenados por relevancia.

Implementar recuperación basada en similitud Jaccard utilizando vectores binarios

In [ ]:
import numpy as np

In [ ]:
# 1. Construir vocabulario global
vocab = sorted(set([t for doc in documents for t in doc]))
vocab_index = {term: i for i, term in enumerate(vocab)}

In [ ]:
# 2. Representar documentos como vectores binarios
doc_vectors = []
for doc in documents:
    vec = np.zeros(len(vocab), dtype=int)
    for term in set(doc):  # conjunto para evitar duplicados
        vec[vocab_index[term]] = 1
    doc_vectors.append(vec)

In [ ]:
# 3. Función para convertir consulta en vector binario
def query_to_vector(query_tokens):
    vec = np.zeros(len(vocab), dtype=int)
    for term in query_tokens:
        if term in vocab_index:
            vec[vocab_index[term]] = 1
    return vec

In [ ]:
# 4. Función de similitud Jaccard
def jaccard_similarity(vec1, vec2):
    intersection = np.sum(np.logical_and(vec1, vec2))
    union = np.sum(np.logical_or(vec1, vec2))
    return intersection / union if union != 0 else 0

In [ ]:
# 5. Recuperación de documentos
def retrieve_jaccard(query_tokens, top_k=5):
    q_vec = query_to_vector(query_tokens)
    scores = []
    for doc_id, d_vec in enumerate(doc_vectors):
        score = jaccard_similarity(q_vec, d_vec)
        scores.append((doc_id, score))
    # Ordenar por score descendente
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return scores[:top_k]

In [ ]:
'''
# Ejemplo de uso
query = ["good", "tast", "product"]  # tokens ya stemmed
results = retrieve_jaccard(query, top_k=5)

for doc_id, score in results:
    print(f"Doc {doc_id} | Score: {score:.3f} | Text: {docs_as_text[doc_id][:80]}...")
'''

Implementar recuperación basada en similitud de coseno utilizando TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# 1. Crear vectorizador TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(docs_as_text)

In [ ]:
# 2. Función para convertir consulta en vector TF-IDF
def query_to_tfidf(query_tokens):
    query_text = " ".join(query_tokens)
    return vectorizer.transform([query_text])

In [ ]:
# 3. Recuperación con similitud de coseno
def retrieve_cosine(query_tokens, top_k=5):
    q_vec = query_to_tfidf(query_tokens)
    # Calcular similitud coseno con todos los documentos
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    # Ordenar por score descendente
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

In [ ]:
'''
# Ejemplo de uso
query = ["good", "tast", "product"]  # tokens ya stemmed
results = retrieve_cosine(query, top_k=5)

for doc_id, score in results:
    print(f"Doc {doc_id} | Score: {score:.3f} | Text: {docs_as_text[doc_id][:80]}...")
'''

Implementar recuperación con BM25.

In [ ]:
import math
from collections import Counter

In [ ]:
# 1. Precalcular estadísticas globales
N = len(documents)  # número de documentos
avgdl = sum(len(doc) for doc in documents) / N

In [ ]:
# 2. Construir frecuencias de términos por documento
doc_freqs = [Counter(doc) for doc in documents]

In [ ]:
# 3. Calcular DF (document frequency) para cada término
df = Counter()
for doc in documents:
    for term in set(doc):
        df[term] += 1

In [ ]:
# 4. Función BM25
def bm25_score(query_tokens, k1=1.5, b=0.75):
    scores = []
    for doc_id, doc in enumerate(documents):
        score = 0
        doc_len = len(doc)
        freqs = doc_freqs[doc_id]
        for term in query_tokens:
            if term not in df:
                continue
            # IDF
            idf = math.log((N - df[term] + 0.5) / (df[term] + 0.5) + 1)
            # TF normalizado
            f = freqs[term]
            denom = f + k1 * (1 - b + b * doc_len / avgdl)
            score += idf * (f * (k1 + 1)) / denom
        scores.append((doc_id, score))
    return sorted(scores, key=lambda x: x[1], reverse=True)

In [ ]:
'''
# Ejemplo de uso
query = ["good", "tast", "product"]  # tokens ya stemmed
results = bm25_score(query, k1=1.5, b=0.75)[:5]

for doc_id, score in results:
    print(f"Doc {doc_id} | Score: {score:.3f} | Text: {docs_as_text[doc_id][:80]}...")
'''

Permitir la ejecución de consultas de texto libre y mostrar un ranking de documentos ordenados por relevancia.

In [ ]:
import re

In [ ]:
# --- Preprocesamiento de consultas ---
def preprocess_query(query_text):
    """
    Limpia y tokeniza la consulta de texto libre, aplicando el mismo pipeline que los documentos.
    """
    query_text = query_text.lower()
    query_text = clean_text(query_text)
    tokens = query_text.split()
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

In [ ]:
# --- Función principal de búsqueda ---
def search(query_text, model="bm25", top_k=5):
    """
    Ejecuta una consulta de texto libre usando el modelo especificado.
    model puede ser: 'jaccard', 'cosine', 'bm25'
    """
    query_tokens = preprocess_query(query_text)

    if model == "jaccard":
        results = retrieve_jaccard(query_tokens, top_k=top_k)
    elif model == "cosine":
        results = retrieve_cosine(query_tokens, top_k=top_k)
    elif model == "bm25":
        results = bm25_score(query_tokens)[:top_k]
    else:
        raise ValueError("Modelo no reconocido. Usa 'jaccard', 'cosine' o 'bm25'.")

    # Mostrar ranking
    print(f"\n=== Ranking con modelo {model.upper()} ===")
    for rank, (doc_id, score) in enumerate(results, start=1):
        print(f"{rank}. Doc {doc_id} | Score: {score:.3f} | Text: {docs_as_text[doc_id][:80]}...")

In [ ]:
'''
# --- Ejemplo de uso ---
query_text = "I love this product, very tasty"
search(query_text, model="jaccard", top_k=5)
search(query_text, model="cosine", top_k=5)
search(query_text, model="bm25", top_k=5)
'''

## c. Interfaz básica
- Interfaz de línea de comandos (CLI).
- Funcionalidades mínimas:
  - Realizar consultas
  - Visualizar los resultados

In [ ]:
!pip install rich

In [ ]:
import time
from IPython.display import clear_output
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.prompt import Prompt, IntPrompt
from rich.layout import Layout
from rich import print as rprint

In [ ]:
console = Console()

def mostrar_encabezado():
    """Muestra el título de la aplicación estilo TUI"""
    console.print(Panel.fit(
        "[bold cyan]SISTEMA DE RECUPERACIÓN DE INFORMACIÓN[/bold cyan]\n"
        "[dim]Amazon Fine Food Reviews[/dim]",
        border_style="blue"
    ))

def mostrar_menu_modelos():
    """Muestra las opciones de modelos disponibles"""
    rprint("[bold yellow]Seleccione el modelo de recuperación:[/bold yellow]")
    rprint("1. [green]BM25[/green] (Probabilístico)")
    rprint("2. [green]Coseno[/green] (Vectorial TF-IDF)")
    rprint("3. [green]Jaccard[/green] (Binario)")
    rprint("4. [red]Salir[/red]")

def renderizar_tabla_resultados(results, model_name):
    """Dibuja una tabla con formato rico para los resultados"""
    if not results:
        console.print(Panel("[bold red]No se encontraron resultados.[/bold red]", border_style="red"))
        return

    table = Table(title=f"Resultados para: {model_name}", show_header=True, header_style="bold magenta")
    table.add_column("Rank", style="dim", width=6)
    table.add_column("Score", justify="right", width=10)
    table.add_column("DocID", width=8)
    table.add_column("Fragmento del Texto", style="cyan")

    for rank, (doc_id, score) in enumerate(results, start=1):
        # Obtener texto original y cortar si es muy largo
        full_text = newsgroupsdocs.iloc[doc_id]["Text"]
        # Limpiamos saltos de línea para que la tabla no se rompa visualmente
        clean_snippet = full_text.replace('\n', ' ')[:100] + "..."

        table.add_row(
            str(rank),
            f"{score:.4f}",
            str(doc_id),
            clean_snippet
        )

    console.print(table)

def main_tui():
    while True:
        clear_output(wait=True) # Limpia la "pantalla" de la celda
        mostrar_encabezado()
        mostrar_menu_modelos()

        # 1. Selección de Modelo
        try:
            opcion = IntPrompt.ask("\n[bold]Opción[/bold]", choices=["1", "2", "3", "4"], default="1")
        except KeyboardInterrupt:
            break

        if opcion == 4:
            rprint("[yellow]Saliendo del sistema...[/yellow]")
            break

        mapa_modelos = {1: "BM25", 2: "Coseno", 3: "Jaccard"}
        modelo_seleccionado = mapa_modelos[opcion]

        # 2. Ingreso de Consulta
        rprint(f"\nModel seleccionado: [bold blue]{modelo_seleccionado}[/bold blue]")
        query_text = Prompt.ask("[bold]Ingrese su consulta[/bold] (ej: 'tasty good product')")

        if not query_text.strip():
            continue

        # 3. Procesamiento (Usando TUS funciones originales)
        with console.status(f"[bold green]Buscando con {modelo_seleccionado}...[/bold green]"):
            # Simular un pequeño delay para que se vea la animación (opcional)
            time.sleep(0.5)

            query_tokens = preprocess_query(query_text)

            results = []
            if modelo_seleccionado == "Jaccard":
                results = retrieve_jaccard(query_tokens, top_k=10)
            elif modelo_seleccionado == "Coseno":
                results = retrieve_cosine(query_tokens, top_k=10)
            elif modelo_seleccionado == "BM25":
                results = bm25_score(query_tokens)[:10]

        # 4. Mostrar Resultados
        clear_output(wait=True)
        mostrar_encabezado()
        rprint(f"Consulta: [italic]'{query_text}'[/italic] | Modelo: [bold]{modelo_seleccionado}[/bold]")
        renderizar_tabla_resultados(results, modelo_seleccionado)

        # Pausa antes de la siguiente búsqueda
        Prompt.ask("\n[dim]Presione Enter para nueva búsqueda...[/dim]")

In [ ]:
'''
if __name__ == "__main__":
    try:
        main_tui()
    except KeyboardInterrupt:
        print("\nSaliendo...")
'''

## d. Evaluación de resultados

- Usar un conjunto de consultas de prueba y documentos relevantes (qrels).
- Calcular para cada consulta:
- Precision
- Recall
- Calcular para todo el sistema:
- MAP

In [ ]:
test_queries = [
    "good quality",
    "bad taste",
    "flavor"
]

In [ ]:
qrels = {}

In [ ]:
for query in test_queries:
    relevant_doc_ids = []
    query_words = query.split()

    # Buscamos en el dataframe original (newsgroupsdocs)
    for idx, row in newsgroupsdocs.iterrows():
        text = row['Text'].lower() # Usamos el texto preprocesado o el original lower
        # Si todas las palabras de la query están en el texto, es relevante
        if all(word in text for word in query_words):
            relevant_doc_ids.append(idx)

    qrels[query] = set(relevant_doc_ids)

In [ ]:
print("=== QRELS (Ground Truth Generado) ===")
for q, docs in qrels.items():
    print(f"Consulta: '{q}' -> {len(docs)} docs relevantes: {list(docs)}")

In [ ]:
def calculate_precision_recall(retrieved_ids, relevant_ids):
    """
    Calcula Precision y Recall para una sola consulta.
    retrieved_ids: Lista de IDs recuperados por el sistema (ordenados).
    relevant_ids: Set de IDs que son verdaderamente relevantes.
    """
    # Intersección: documentos recuperados que son relevantes
    true_positives = len([doc for doc in retrieved_ids if doc in relevant_ids])

    # Precision = Relevantes Recuperados / Total Recuperados
    precision = true_positives / len(retrieved_ids) if retrieved_ids else 0

    # Recall = Relevantes Recuperados / Total Relevantes en la colección
    recall = true_positives / len(relevant_ids) if relevant_ids else 0

    return precision, recall

In [ ]:
def calculate_ap(retrieved_ids, relevant_ids):
    """
    Calcula Average Precision (AP) para una consulta.
    Premia que los relevantes aparezcan PRIMERO en la lista.
    """
    if not relevant_ids:
        return 0.0

    score_sum = 0.0
    hits = 0

    for i, doc_id in enumerate(retrieved_ids):
        # Si el documento en la posición 'i' es relevante
        if doc_id in relevant_ids:
            hits += 1
            # Calculamos la precisión en este punto de corte (Precision@k)
            p_at_k = hits / (i + 1)
            score_sum += p_at_k

    # Average Precision es el promedio de las precisiones en cada acierto
    return score_sum / len(relevant_ids)

In [ ]:
def evaluate_system(model_name, top_k=10):
    metrics_per_query = []
    ap_sum = 0

    print(f"\nEvaluating Model: {model_name} (Top {top_k})")

    for query in test_queries:
        # 1. Preprocesar y Buscar
        q_tokens = preprocess_query(query) # Tu función existente

        # Selección del modelo usando tus funciones existentes
        results = []
        if model_name == "Jaccard":
            results = retrieve_jaccard(q_tokens, top_k=top_k)
        elif model_name == "Coseno":
            results = retrieve_cosine(q_tokens, top_k=top_k)
        elif model_name == "BM25":
            results = bm25_score(q_tokens)[:top_k]

        # Extraer solo los IDs de los resultados
        retrieved_ids = [doc_id for doc_id, score in results]
        relevant_ids = qrels[query]

        # 2. Calcular Métricas
        p, r = calculate_precision_recall(retrieved_ids, relevant_ids)
        ap = calculate_ap(retrieved_ids, relevant_ids)

        ap_sum += ap

        metrics_per_query.append({
            "Query": query,
            "Precision": round(p, 4),
            "Recall": round(r, 4),
            "AP": round(ap, 4),
            "Retrieved": retrieved_ids,
            "Relevant": list(relevant_ids)
        })

    # 3. Calcular MAP (Promedio del AP de todas las consultas)
    mean_average_precision = ap_sum / len(test_queries)

    # Mostrar tabla por consulta
    df_res = pd.DataFrame(metrics_per_query)
    display(df_res[["Query", "Precision", "Recall", "AP"]])

    print(f"--> MAP (Mean Average Precision) para {model_name}: {mean_average_precision:.4f}")
    return mean_average_precision

In [ ]:
map_jaccard = evaluate_system("Jaccard", top_k=10)
map_cosine = evaluate_system("Coseno", top_k=10)
map_bm25 = evaluate_system("BM25", top_k=10)

In [ ]:
# Tabla resumen final
comparison = pd.DataFrame({
    "Modelo": ["Jaccard", "Coseno", "BM25"],
    "MAP": [map_jaccard, map_cosine, map_bm25]
})

In [ ]:
print("\n=== RANKING FINAL DE MODELOS ===")
display(comparison.sort_values(by="MAP", ascending=False))

# Producto Final

In [ ]:
if __name__ == "__main__":
    try:
        main_tui()
    except KeyboardInterrupt:
        print("\nSaliendo...")